<a href="https://colab.research.google.com/github/Shamima21/CancerGeneIdentification/blob/main/Enrichment%20Analysis/Enrichment_Analysis_by_Enrichr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gseapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.9/689.9 kB 9.4 MB/s eta 0:00:00


In [8]:
 pip install gseapy pandas openpyxl

In [10]:
"""
Enrichr Full-Library Enrichment Analysis
========================================

Install:
    pip install gseapy pandas openpyxl
"""

from pathlib import Path
import re

import gseapy as gp
import pandas as pd


ADJUSTED_P_THRESHOLD = 0.05


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_gene_list(raw_text):
    """
    Clean comma-, semicolon-, space-, tab-, or newline-separated genes.
    """
    normalized_text = (
        raw_text
        .replace(",", " ")
        .replace(";", " ")
        .replace("\n", " ")
        .replace("\t", " ")
    )

    genes = [
        gene.strip().upper()
        for gene in normalized_text.split()
        if gene.strip()
    ]

    return list(dict.fromkeys(genes))


def create_safe_name(text):
    """
    Create a safe file or folder name.
    """
    safe_name = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        text.strip(),
    )

    return safe_name or "Cancer_Dataset"


def convert_numeric_columns(results_df):
    """
    Convert statistical result columns to numeric values.
    """
    numeric_columns = [
        "P-value",
        "Adjusted P-value",
        "Old P-value",
        "Old Adjusted P-value",
        "Odds Ratio",
        "Combined Score",
    ]

    for column in numeric_columns:
        if column in results_df.columns:
            results_df[column] = pd.to_numeric(
                results_df[column],
                errors="coerce",
            )

    return results_df


# ============================================================
# GET ALL ENRICHR LIBRARIES
# ============================================================

def get_all_enrichr_libraries():
    """
    Retrieve all currently available human Enrichr libraries.
    """
    print("\nRetrieving all available human Enrichr libraries...")

    libraries = gp.get_library_name(
        organism="human"
    )

    if not libraries:
        raise RuntimeError(
            "No Enrichr libraries could be retrieved."
        )

    libraries = sorted(set(libraries))

    print(
        f"\nTotal available Enrichr libraries: "
        f"{len(libraries)}"
    )

    return libraries


def display_libraries(libraries):
    """
    Display all available libraries with index numbers.
    """
    print("\n" + "=" * 70)
    print("AVAILABLE HUMAN ENRICHR LIBRARIES")
    print("=" * 70)

    for index, library in enumerate(
        libraries,
        start=1,
    ):
        print(f"{index:3d}. {library}")


# ============================================================
# LIBRARY SELECTION
# ============================================================

def select_libraries(all_libraries):
    """
    Allow the user to run all libraries or selected libraries.
    """
    print("\nLibrary options:")
    print("1. Run all available Enrichr libraries")
    print("2. Select libraries by number")
    print("3. Use the paper-specific libraries")

    choice = input(
        "Select an option [1/2/3]: "
    ).strip()

    if choice == "1":
        return all_libraries

    if choice == "2":
        selection = input(
            "\nEnter library numbers separated by commas "
            "(example: 1,5,10):\n"
        ).strip()

        if not selection:
            raise ValueError(
                "No library numbers were entered."
            )

        selected_indices = []

        for value in selection.split(","):
            value = value.strip()

            if not value.isdigit():
                raise ValueError(
                    f"Invalid library number: {value}"
                )

            index = int(value)

            if index < 1 or index > len(all_libraries):
                raise ValueError(
                    f"Library number {index} is outside "
                    "the available range."
                )

            selected_indices.append(index - 1)

        selected_libraries = [
            all_libraries[index]
            for index in selected_indices
        ]

        return list(dict.fromkeys(selected_libraries))

    if choice in {"", "3"}:
        paper_libraries = [
            "DisGeNET",
            "GO_Biological_Process_2023",
            "Reactome_2022",
            "KEGG_2021_Human",
            "WikiPathway_2023_Human",
            "Jensen_TISSUES",
            "CCLE_Proteomics_2020",
            "Cancer_Cell_Line_Encyclopedia",
            "NCI-60_Cancer_Cell_Lines",
        ]

        selected_libraries = [
            library
            for library in paper_libraries
            if library in all_libraries
        ]

        unavailable = [
            library
            for library in paper_libraries
            if library not in all_libraries
        ]

        if unavailable:
            print("\nCurrently unavailable paper libraries:")

            for library in unavailable:
                print(f"  - {library}")

        if not selected_libraries:
            raise RuntimeError(
                "None of the paper-specific libraries are available."
            )

        return selected_libraries

    raise ValueError(
        "Invalid option. Select 1, 2, or 3."
    )


# ============================================================
# RUN ENRICHR
# ============================================================

def run_enrichr(
    genes,
    selected_libraries,
    output_directory,
):
    """
    Run online Enrichr analysis with the default Enrichr background.
    """
    print("\nLibraries selected for analysis:")

    for library in selected_libraries:
        print(f"  - {library}")

    print("\nRunning Enrichr analysis...")

    enrichment = gp.enrichr(
        gene_list=genes,
        gene_sets=selected_libraries,
        organism="human",
        outdir=str(
            output_directory / "Enrichr_raw_output"
        ),
        cutoff=ADJUSTED_P_THRESHOLD,
        no_plot=True,
        verbose=True,
    )

    if enrichment.results is None:
        return pd.DataFrame()

    results_df = enrichment.results.copy()

    if results_df.empty:
        return results_df

    results_df = convert_numeric_columns(
        results_df
    )

    sorting_columns = []
    sorting_order = []

    if "Adjusted P-value" in results_df.columns:
        sorting_columns.append(
            "Adjusted P-value"
        )
        sorting_order.append(True)

    if "Combined Score" in results_df.columns:
        sorting_columns.append(
            "Combined Score"
        )
        sorting_order.append(False)

    if sorting_columns:
        results_df = results_df.sort_values(
            by=sorting_columns,
            ascending=sorting_order,
        )

    return results_df.reset_index(drop=True)


# ============================================================
# LIBRARY-WISE RESULT SUMMARY
# ============================================================

def create_library_result_summary(
    selected_libraries,
    complete_results,
):
    """
    Show whether each selected library returned results.
    No score calculation is performed.
    """
    if complete_results.empty:
        return pd.DataFrame(
            {
                "Library": selected_libraries,
                "Returned Terms": 0,
                "Significant Terms": 0,
            }
        )

    if "Gene_set" not in complete_results.columns:
        raise ValueError(
            "The Gene_set column was not found."
        )

    total_counts = (
        complete_results
        .groupby("Gene_set")
        .size()
        .to_dict()
    )

    if "Adjusted P-value" in complete_results.columns:
        significant_results = complete_results[
            complete_results["Adjusted P-value"]
            < ADJUSTED_P_THRESHOLD
        ]

        significant_counts = (
            significant_results
            .groupby("Gene_set")
            .size()
            .to_dict()
        )
    else:
        significant_counts = {}

    summary_rows = []

    for library in selected_libraries:
        summary_rows.append(
            {
                "Library": library,
                "Returned Terms": total_counts.get(
                    library,
                    0,
                ),
                "Significant Terms": significant_counts.get(
                    library,
                    0,
                ),
            }
        )

    return pd.DataFrame(summary_rows)


# ============================================================
# SAVE RESULTS
# ============================================================

def save_results(
    dataset_name,
    genes,
    selected_libraries,
    complete_results,
    output_directory,
):
    """
    Save selected genes, library list, complete results,
    significant results, and library access summary.
    """
    selected_genes_df = pd.DataFrame(
        {
            "Rank": range(1, len(genes) + 1),
            "Gene": genes,
        }
    )

    selected_libraries_df = pd.DataFrame(
        {
            "Library": selected_libraries
        }
    )

    if (
        not complete_results.empty
        and "Adjusted P-value" in complete_results.columns
    ):
        significant_results = complete_results[
            complete_results["Adjusted P-value"]
            < ADJUSTED_P_THRESHOLD
        ].copy()
    else:
        significant_results = pd.DataFrame()

    library_summary = create_library_result_summary(
        selected_libraries,
        complete_results,
    )

    selected_genes_df.to_csv(
        output_directory / "selected_genes.csv",
        index=False,
    )

    selected_libraries_df.to_csv(
        output_directory / "selected_libraries.csv",
        index=False,
    )

    complete_results.to_csv(
        output_directory / "complete_enrichment_results.csv",
        index=False,
    )

    significant_results.to_csv(
        output_directory / "significant_enrichment_results.csv",
        index=False,
    )

    library_summary.to_csv(
        output_directory / "library_access_summary.csv",
        index=False,
    )

    complete_results.to_csv(
        output_directory / "enrichr_results.txt",
        sep="\t",
        index=False,
    )

    excel_file = (
        output_directory
        / f"{dataset_name}_Enrichr_results.xlsx"
    )

    with pd.ExcelWriter(
        excel_file,
        engine="openpyxl",
    ) as writer:

        selected_genes_df.to_excel(
            writer,
            sheet_name="Selected Genes",
            index=False,
        )

        selected_libraries_df.to_excel(
            writer,
            sheet_name="Libraries",
            index=False,
        )

        complete_results.to_excel(
            writer,
            sheet_name="Complete Results",
            index=False,
        )

        significant_results.to_excel(
            writer,
            sheet_name="Significant Results",
            index=False,
        )

        library_summary.to_excel(
            writer,
            sheet_name="Library Access",
            index=False,
        )

    return significant_results, library_summary, excel_file


# ============================================================
# MAIN WORKFLOW
# ============================================================

def main():
    print("=" * 70)
    print("ENRICHR FULL-LIBRARY ENRICHMENT ANALYSIS")
    print("=" * 70)

    dataset_name = input(
        "\nEnter the dataset name: "
    ).strip()

    if not dataset_name:
        dataset_name = "Cancer_Dataset"

    safe_dataset_name = create_safe_name(
        dataset_name
    )

    gene_input = input(
        "\nEnter the final ranked genes, separated "
        "by commas or spaces:\n"
    )

    genes = clean_gene_list(
        gene_input
    )

    if not genes:
        raise ValueError(
            "No valid gene symbols were entered."
        )

    print(
        f"\nNumber of selected genes: {len(genes)}"
    )

    print("Selected genes:")
    print(", ".join(genes))

    output_directory = Path(
        f"{safe_dataset_name}_Enrichr_results"
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    all_libraries = get_all_enrichr_libraries()

    display_libraries(
        all_libraries
    )

    selected_libraries = select_libraries(
        all_libraries
    )

    complete_results = run_enrichr(
        genes=genes,
        selected_libraries=selected_libraries,
        output_directory=output_directory,
    )

    if complete_results.empty:
        print(
            "\nNo enrichment results were returned."
        )
        return

    (
        significant_results,
        library_summary,
        excel_file,
    ) = save_results(
        dataset_name=safe_dataset_name,
        genes=genes,
        selected_libraries=selected_libraries,
        complete_results=complete_results,
        output_directory=output_directory,
    )

    print("\n" + "=" * 70)
    print("ENRICHR ANALYSIS COMPLETED")
    print("=" * 70)

    print(
        f"Selected libraries: "
        f"{len(selected_libraries)}"
    )

    print(
        f"Total returned terms: "
        f"{len(complete_results)}"
    )

    print(
        "Significant terms "
        f"(Adjusted P-value < {ADJUSTED_P_THRESHOLD}): "
        f"{len(significant_results)}"
    )

    print("\nLibrary access summary:")
    print(
        library_summary.to_string(
            index=False
        )
    )

    print(
        f"\nExcel workbook saved as:\n"
        f"{excel_file}"
    )

    print(
        f"\nAll files saved in:\n"
        f"{output_directory.resolve()}"
    )


if __name__ == "__main__":
    try:
        main()

    except KeyboardInterrupt:
        print("\nAnalysis was cancelled.")

    except Exception as error:
        print(f"\nAnalysis failed: {error}")

ENRICHR FULL-LIBRARY ENRICHMENT ANALYSIS

Enter the dataset name: DLBCL

Enter the final ranked genes, separated by commas or spaces:
MCM2, MCM3, MCM6, MCM7, CDK1, KIF11, CCNB1, CDC20, BUB1B, TOP2A, MKI67, CENPF, NEK2, AURKA, PLK1, CKS1B, CKS2, CCNA2, MAD2L1, PCNA, CHEK1, E2F1, FOXM1, TYMS, RRM2, UBE2C, NUSAP1, TPX2, BIRC5, ASPM

Number of selected genes: 30
Selected genes:
MCM2, MCM3, MCM6, MCM7, CDK1, KIF11, CCNB1, CDC20, BUB1B, TOP2A, MKI67, CENPF, NEK2, AURKA, PLK1, CKS1B, CKS2, CCNA2, MAD2L1, PCNA, CHEK1, E2F1, FOXM1, TYMS, RRM2, UBE2C, NUSAP1, TPX2, BIRC5, ASPM

Retrieving all available human Enrichr libraries...

Total available Enrichr libraries: 228

AVAILABLE HUMAN ENRICHR LIBRARIES
  1. ARCHS4_Cell-lines
  2. ARCHS4_IDG_Coexp
  3. ARCHS4_Kinases_Coexp
  4. ARCHS4_TFs_Coexp
  5. ARCHS4_Tissues
  6. Achilles_fitness_decrease
  7. Achilles_fitness_increase
  8. Aging_Perturbations_from_GEO_down
  9. Aging_Perturbations_from_GEO_up
 10. Allen_Brain_Atlas_10x_scRNA_2021
 11. Alle

2026-07-30 10:22:52,260 [INFO] Online enrichment analysis with libraries: ARCHS4_Cell-lines, ARCHS4_IDG_Coexp, ARCHS4_Kinases_Coexp, ARCHS4_TFs_Coexp, ARCHS4_Tissues, Achilles_fitness_decrease, Achilles_fitness_increase, Aging_Perturbations_from_GEO_down, Aging_Perturbations_from_GEO_up, Allen_Brain_Atlas_10x_scRNA_2021, Allen_Brain_Atlas_down, Allen_Brain_Atlas_up, Azimuth_2023, Azimuth_Cell_Types_2021, BioCarta_2013, BioCarta_2015, BioCarta_2016, BioPlanet_2019, BioPlex_2017, CCLE_Proteomics_2020, CM4AI_U2OS_Protein_Localization_Assemblies, COMPARTMENTS_Curated_2025, COMPARTMENTS_Experimental_2025, CORUM, COVID-19_Related_Gene_Sets, COVID-19_Related_Gene_Sets_2021, Cancer_Cell_Line_Encyclopedia, Carcinogenome, CellMarker_2024, CellMarker_Augmented_2021, ChEA_2013, ChEA_2015, ChEA_2016, ChEA_2022, Chromosome_Location, Chromosome_Location_hg19, ClinVar_2019, ClinVar_2025, DGIdb_Drug_Targets_2024, DSigDB, Data_Acquisition_Method_Most_Popular_Genes, DepMap_CRISPR_GeneDependency_CellLines


ENRICHR ANALYSIS COMPLETED
Selected libraries: 228
Total returned terms: 108667
Significant terms (Adjusted P-value < 0.05): 40516

Library access summary:
                                           Library  Returned Terms  Significant Terms
                                 ARCHS4_Cell-lines              68                  2
                                  ARCHS4_IDG_Coexp              28                 11
                              ARCHS4_Kinases_Coexp              74                 48
                                  ARCHS4_TFs_Coexp             418                201
                                    ARCHS4_Tissues              45                 18
                         Achilles_fitness_decrease              84                 25
                         Achilles_fitness_increase              69                  8
                 Aging_Perturbations_from_GEO_down              74                 25
                   Aging_Perturbations_from_GEO_up              41   

In [9]:
"""
CGI_MMLP Structured Biological Relevance Assessment
====================================================

Required packages:
    pip install gseapy pandas openpyxl
"""

from pathlib import Path
import re

import gseapy as gp
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

ADJUSTED_P_THRESHOLD = 0.05
TOP_TERMS_TO_SAVE = 30

REQUESTED_LIBRARIES = [
    "DisGeNET",
    "GO_Biological_Process_2023",
    "Reactome_2022",
    "KEGG_2021_Human",
    "WikiPathway_2023_Human",
    "Jensen_TISSUES",
    "CCLE_Proteomics_2020",
    "Cancer_Cell_Line_Encyclopedia",
    "NCI-60_Cancer_Cell_Lines",
]

CANCER_RELEVANT_LIBRARIES = {
    "DisGeNET",
    "CCLE_Proteomics_2020",
    "Cancer_Cell_Line_Encyclopedia",
    "NCI-60_Cancer_Cell_Lines",
}

CANCER_KEYWORDS = {
    "cancer",
    "carcinoma",
    "tumor",
    "tumour",
    "neoplasm",
    "neoplastic",
    "malignant",
    "malignancy",
    "metastasis",
    "metastatic",
    "lymphoma",
    "leukemia",
    "leukaemia",
    "prostate",
    "breast",
    "lung",
    "melanoma",
    "glioma",
    "myeloma",
    "sarcoma",
    "oncogenic",
    "oncogene",
    "cell cycle",
    "dna replication",
    "dna repair",
    "apoptosis",
    "angiogenesis",
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_gene_list(raw_text):
    normalized_text = (
        raw_text
        .replace(",", " ")
        .replace(";", " ")
        .replace("\n", " ")
        .replace("\t", " ")
    )

    genes = [
        gene.strip().upper()
        for gene in normalized_text.split()
        if gene.strip()
    ]

    return list(dict.fromkeys(genes))


def create_safe_name(text):
    safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", text.strip())
    return safe_name or "Cancer_Dataset"


def contains_cancer_keyword(term):
    normalized_term = str(term).lower()

    return any(
        keyword in normalized_term
        for keyword in CANCER_KEYWORDS
    )


def convert_numeric_columns(results_df):
    numeric_columns = [
        "P-value",
        "Adjusted P-value",
        "Old P-value",
        "Old Adjusted P-value",
        "Odds Ratio",
        "Combined Score",
    ]

    for column in numeric_columns:
        if column in results_df.columns:
            results_df[column] = pd.to_numeric(
                results_df[column],
                errors="coerce",
            )

    return results_df


def get_top_terms(results_df, number_of_terms):
    if results_df.empty:
        return pd.DataFrame()

    sorting_columns = ["Adjusted P-value"]
    sorting_order = [True]

    if "Combined Score" in results_df.columns:
        sorting_columns.append("Combined Score")
        sorting_order.append(False)

    return (
        results_df
        .sort_values(
            by=sorting_columns,
            ascending=sorting_order,
        )
        .head(number_of_terms)
        .copy()
    )


# ============================================================
# LIBRARY VALIDATION
# ============================================================

def validate_libraries(requested_libraries):
    print("\nChecking available Enrichr libraries...")

    available_libraries = set(
        gp.get_library_name(organism="human")
    )

    valid_libraries = [
        library
        for library in requested_libraries
        if library in available_libraries
    ]

    unavailable_libraries = [
        library
        for library in requested_libraries
        if library not in available_libraries
    ]

    if unavailable_libraries:
        print("\nUnavailable or renamed libraries:")

        for library in unavailable_libraries:
            print(f"  - {library}")

    if not valid_libraries:
        raise RuntimeError(
            "None of the requested Enrichr libraries are available."
        )

    print("\nLibraries included in the analysis:")

    for library in valid_libraries:
        print(f"  - {library}")

    return valid_libraries


# ============================================================
# ENRICHR ANALYSIS
# ============================================================

def run_enrichr_analysis(
    genes,
    libraries,
    output_directory,
):
    print("\nRunning Enrichr analysis...")

    raw_output_directory = (
        output_directory / "Enrichr_raw_output"
    )

    raw_output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    enrichment = gp.enrichr(
        gene_list=genes,
        gene_sets=libraries,
        organism="human",
        outdir=str(raw_output_directory),
        cutoff=ADJUSTED_P_THRESHOLD,
        no_plot=True,
        verbose=True,
    )

    if enrichment.results is None:
        return pd.DataFrame()

    results_df = enrichment.results.copy()

    if results_df.empty:
        return results_df

    results_df = convert_numeric_columns(results_df)

    if "Adjusted P-value" not in results_df.columns:
        raise ValueError(
            "Adjusted P-value column was not found."
        )

    sorting_columns = ["Adjusted P-value"]
    sorting_order = [True]

    if "Combined Score" in results_df.columns:
        sorting_columns.append("Combined Score")
        sorting_order.append(False)

    return (
        results_df
        .sort_values(
            by=sorting_columns,
            ascending=sorting_order,
        )
        .reset_index(drop=True)
    )


# ============================================================
# SIGNIFICANT RESULTS
# ============================================================

def filter_significant_results(results_df):
    significant_results = results_df[
        results_df["Adjusted P-value"]
        < ADJUSTED_P_THRESHOLD
    ].copy()

    return (
        significant_results
        .sort_values(
            by="Adjusted P-value",
            ascending=True,
        )
        .reset_index(drop=True)
    )


# ============================================================
# PRR AND EQSCORE
# ============================================================

def calculate_prr_and_eqscore(significant_results):
    annotated_results = significant_results.copy()

    if annotated_results.empty:
        summary = pd.DataFrame(
            {
                "Metric": [
                    "Total significant terms",
                    "Cancer-library significant terms",
                    "Cancer-keyword significant terms",
                    "PRR (%)",
                    "EQScore (%)",
                ],
                "Value": [0, 0, 0, 0.0, 0.0],
            }
        )

        return annotated_results, summary

    annotated_results["Cancer-Relevant Library"] = (
        annotated_results["Gene_set"].isin(
            CANCER_RELEVANT_LIBRARIES
        )
    )

    annotated_results["Cancer-Related Term"] = (
        annotated_results["Term"].apply(
            contains_cancer_keyword
        )
    )

    total_significant_terms = len(annotated_results)

    cancer_library_terms = int(
        annotated_results[
            "Cancer-Relevant Library"
        ].sum()
    )

    cancer_keyword_terms = int(
        annotated_results[
            "Cancer-Related Term"
        ].sum()
    )

    prr = (
        cancer_library_terms
        / total_significant_terms
        * 100
    )

    eqscore = (
        cancer_keyword_terms
        / total_significant_terms
        * 100
    )

    summary = pd.DataFrame(
        {
            "Metric": [
                "Total significant terms",
                "Cancer-library significant terms",
                "Cancer-keyword significant terms",
                "PRR (%)",
                "EQScore (%)",
            ],
            "Value": [
                total_significant_terms,
                cancer_library_terms,
                cancer_keyword_terms,
                round(prr, 2),
                round(eqscore, 2),
            ],
        }
    )

    return annotated_results, summary


# ============================================================
# LIBRARY-WISE SUMMARY
# ============================================================

def create_library_summary(
    complete_results,
    significant_results,
):
    complete_summary = (
        complete_results
        .groupby("Gene_set")
        .agg(
            Total_Enriched_Terms=("Term", "count"),
            Minimum_P_Value=("P-value", "min"),
            Minimum_Adjusted_P_Value=(
                "Adjusted P-value",
                "min",
            ),
        )
    )

    if significant_results.empty:
        significant_counts = pd.Series(
            0,
            index=complete_summary.index,
            name="Significant_Terms",
        )
    else:
        significant_counts = (
            significant_results
            .groupby("Gene_set")
            .size()
            .rename("Significant_Terms")
        )

    library_summary = complete_summary.join(
        significant_counts,
        how="left",
    )

    library_summary["Significant_Terms"] = (
        library_summary["Significant_Terms"]
        .fillna(0)
        .astype(int)
    )

    library_summary["Significant_Proportion_Percent"] = (
        library_summary["Significant_Terms"]
        / library_summary["Total_Enriched_Terms"]
        * 100
    ).round(2)

    return library_summary.reset_index()


# ============================================================
# GENE-PATHWAY ASSOCIATIONS
# ============================================================

def create_gene_pathway_table(significant_results):
    if significant_results.empty:
        return pd.DataFrame()

    if "Genes" not in significant_results.columns:
        return pd.DataFrame()

    selected_columns = [
        column
        for column in [
            "Gene_set",
            "Term",
            "Adjusted P-value",
            "Combined Score",
            "Genes",
        ]
        if column in significant_results.columns
    ]

    associations = significant_results[
        selected_columns
    ].copy()

    associations["Gene"] = (
        associations["Genes"]
        .astype(str)
        .str.replace(",", ";", regex=False)
        .str.split(";")
    )

    associations = associations.explode("Gene")

    associations["Gene"] = (
        associations["Gene"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    associations = associations[
        associations["Gene"] != ""
    ]

    return (
        associations
        .drop(columns=["Genes"])
        .drop_duplicates()
        .reset_index(drop=True)
    )


# ============================================================
# SAVE RESULTS
# ============================================================

def save_all_results(
    dataset_name,
    genes,
    complete_results,
    significant_results,
    library_summary,
    score_summary,
    gene_pathway_table,
    output_directory,
):
    selected_genes_df = pd.DataFrame(
        {
            "Rank": range(1, len(genes) + 1),
            "Gene": genes,
        }
    )

    top_terms = get_top_terms(
        significant_results,
        TOP_TERMS_TO_SAVE,
    )

    selected_genes_df.to_csv(
        output_directory / "selected_ranked_genes.csv",
        index=False,
    )

    complete_results.to_csv(
        output_directory / "complete_enrichment_results.csv",
        index=False,
    )

    significant_results.to_csv(
        output_directory / "significant_enrichment_results.csv",
        index=False,
    )

    top_terms.to_csv(
        output_directory / "top_significant_terms.csv",
        index=False,
    )

    library_summary.to_csv(
        output_directory / "library_wise_summary.csv",
        index=False,
    )

    score_summary.to_csv(
        output_directory / "prr_eqscore_summary.csv",
        index=False,
    )

    gene_pathway_table.to_csv(
        output_directory / "gene_pathway_associations.csv",
        index=False,
    )

    complete_results.to_csv(
        output_directory / "enrichr_results.txt",
        sep="\t",
        index=False,
    )

    excel_file = (
        output_directory
        / f"{dataset_name}_complete_workflow.xlsx"
    )

    with pd.ExcelWriter(
        excel_file,
        engine="openpyxl",
    ) as writer:

        selected_genes_df.to_excel(
            writer,
            sheet_name="Selected Genes",
            index=False,
        )

        complete_results.to_excel(
            writer,
            sheet_name="Complete Results",
            index=False,
        )

        significant_results.to_excel(
            writer,
            sheet_name="Significant Results",
            index=False,
        )

        top_terms.to_excel(
            writer,
            sheet_name="Top Terms",
            index=False,
        )

        library_summary.to_excel(
            writer,
            sheet_name="Library Summary",
            index=False,
        )

        score_summary.to_excel(
            writer,
            sheet_name="PRR EQScore",
            index=False,
        )

        gene_pathway_table.to_excel(
            writer,
            sheet_name="Gene Pathway",
            index=False,
        )

    print(f"\nExcel workbook saved as: {excel_file}")


# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 70)
    print(
        "CGI_MMLP STRUCTURED BIOLOGICAL "
        "RELEVANCE ASSESSMENT"
    )
    print("=" * 70)

    dataset_name = input(
        "\nEnter the dataset name, e.g., "
        "DLBCL, Leukemia, or Prostate: "
    ).strip()

    if not dataset_name:
        dataset_name = "Cancer_Dataset"

    safe_dataset_name = create_safe_name(dataset_name)

    gene_input = input(
        "\nEnter the final ranked genes, separated by "
        "commas or spaces:\n"
    )

    genes = clean_gene_list(gene_input)

    if not genes:
        raise ValueError(
            "No valid gene symbols were entered."
        )

    print(f"\nNumber of selected genes: {len(genes)}")
    print("Selected genes:")
    print(", ".join(genes))

    output_directory = Path(
        f"{safe_dataset_name}_CGI_MMLP_results"
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    valid_libraries = validate_libraries(
        REQUESTED_LIBRARIES
    )

    complete_results = run_enrichr_analysis(
        genes=genes,
        libraries=valid_libraries,
        output_directory=output_directory,
    )

    if complete_results.empty:
        print("\nNo enrichment results were returned.")
        return

    significant_results = filter_significant_results(
        complete_results
    )

    (
        significant_results,
        score_summary,
    ) = calculate_prr_and_eqscore(
        significant_results
    )

    library_summary = create_library_summary(
        complete_results,
        significant_results,
    )

    gene_pathway_table = create_gene_pathway_table(
        significant_results
    )

    save_all_results(
        dataset_name=safe_dataset_name,
        genes=genes,
        complete_results=complete_results,
        significant_results=significant_results,
        library_summary=library_summary,
        score_summary=score_summary,
        gene_pathway_table=gene_pathway_table,
        output_directory=output_directory,
    )

    print("\n" + "=" * 70)
    print("WORKFLOW COMPLETED")
    print("=" * 70)

    print(f"Dataset: {dataset_name}")
    print(f"Selected genes: {len(genes)}")
    print(f"Total enriched terms: {len(complete_results)}")
    print(
        "Significant terms "
        f"(Adjusted P-value < {ADJUSTED_P_THRESHOLD}): "
        f"{len(significant_results)}"
    )

    print("\nPRR and EQScore:")
    print(score_summary.to_string(index=False))

    print(
        "\nResults saved in:\n"
        f"{output_directory.resolve()}"
    )


if __name__ == "__main__":
    try:
        main()

    except KeyboardInterrupt:
        print("\nAnalysis was cancelled.")

    except Exception as error:
        print(f"\nAnalysis failed: {error}")

CGI_MMLP STRUCTURED BIOLOGICAL RELEVANCE ASSESSMENT

Enter the dataset name, e.g., DLBCL, Leukemia, or Prostate:  DLBCL

Enter the final ranked genes, separated by commas or spaces:
MCM2, MCM3, MCM6, MCM7, CDK1, KIF11, CCNB1, CDC20, BUB1B, TOP2A, MKI67, CENPF, NEK2, AURKA, PLK1, CKS1B, CKS2, CCNA2, MAD2L1, PCNA, CHEK1, E2F1, FOXM1, TYMS, RRM2, UBE2C, NUSAP1, TPX2, BIRC5, ASPM

Number of selected genes: 30
Selected genes:
MCM2, MCM3, MCM6, MCM7, CDK1, KIF11, CCNB1, CDC20, BUB1B, TOP2A, MKI67, CENPF, NEK2, AURKA, PLK1, CKS1B, CKS2, CCNA2, MAD2L1, PCNA, CHEK1, E2F1, FOXM1, TYMS, RRM2, UBE2C, NUSAP1, TPX2, BIRC5, ASPM

Checking available Enrichr libraries...

Libraries included in the analysis:
  - DisGeNET
  - GO_Biological_Process_2023
  - Reactome_2022
  - KEGG_2021_Human
  - WikiPathway_2023_Human
  - Jensen_TISSUES
  - CCLE_Proteomics_2020
  - Cancer_Cell_Line_Encyclopedia
  - NCI-60_Cancer_Cell_Lines

Running Enrichr analysis...


2026-07-30 10:19:36,113 [INFO] Online enrichment analysis with libraries: DisGeNET, GO_Biological_Process_2023, Reactome_2022, KEGG_2021_Human, WikiPathway_2023_Human, Jensen_TISSUES, CCLE_Proteomics_2020, Cancer_Cell_Line_Encyclopedia, NCI-60_Cancer_Cell_Lines
2026-07-30 10:19:36,967 [INFO] Save enrichment results for Enrichr
2026-07-30 10:19:36,968 [INFO] Done.



Excel workbook saved as: DLBCL_CGI_MMLP_results/DLBCL_complete_workflow.xlsx

WORKFLOW COMPLETED
Dataset: DLBCL
Selected genes: 30
Total enriched terms: 2274
Significant terms (Adjusted P-value < 0.05): 891

PRR and EQScore:
                          Metric  Value
         Total significant terms 891.00
Cancer-library significant terms 449.00
Cancer-keyword significant terms 342.00
                         PRR (%)  50.39
                     EQScore (%)  38.38

Results saved in:
/content/DLBCL_CGI_MMLP_results
